# 03 — Recommender training
Trains the two-tower model on synthetic interactions and computes hit@10 and nDCG@10.

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
for sub in ('packages/core', 'packages/ml'):
    p = os.path.join(ROOT, sub)
    if p not in sys.path: sys.path.insert(0, p)
import numpy as np
import torch
from echomind_ml.recommender import TrainConfig, _build_synthetic, train

In [ ]:
user_idx, train_content, all_books = _build_synthetic(num_users=200, num_books=80, content_dim=64)
num_users = int(user_idx.max() + 1)
print('interactions:', len(user_idx), 'users:', num_users, 'books:', len(all_books))

In [ ]:
model = train(user_idx, train_content, num_users=num_users, cfg=TrainConfig(epochs=3, batch_size=128))

In [ ]:
with torch.no_grad():
    item_emb = model.item(torch.from_numpy(all_books.astype(np.float32))).numpy()
    user_emb = model.user(torch.arange(num_users)).numpy()

# evaluation: for each user, rank their next held-out book
from collections import defaultdict
user_books = defaultdict(list)
for u, c in zip(user_idx, train_content):
    sim = (c @ all_books.T)
    user_books[int(u)].append(int(sim.argmax()))
hits, ndcgs = [], []
for u, true_books in user_books.items():
    scores = user_emb[u] @ item_emb.T
    ranked = np.argsort(-scores)
    rank = np.where(np.isin(ranked, true_books))[0][0]
    hits.append(int(rank < 10))
    ndcgs.append(1.0 / np.log2(rank + 2))
print(f'hit@10: {np.mean(hits):.3f}   ndcg@10: {np.mean(ndcgs):.3f}')